# Dia 1 — Preparacao de Dados e Analise Exploratoria (EDA)

**Autor:** Henry  
**Dataset:** Olist Brazilian E-Commerce  
**Objetivo:** Construir o DataFrame unificado de pedidos entregues, criar a variavel alvo `foi_atraso`, e analisar correlacoes de Pearson para identificar features promissoras para o modelo de ML.

---

## Task 1 — Carregar CSVs, Filtrar Pedidos Delivered e Realizar Merges

Carregamos 5 tabelas do dataset Olist e unificamos via left joins sequenciais.  
Apenas pedidos com `order_status == 'delivered'` sao mantidos.

In [1]:
# Setup: imports e constantes
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

DATA_RAW = '../data/raw/'
print('Setup OK')

Setup OK


In [2]:
# 1. Carregar orders
orders = pd.read_csv(DATA_RAW + 'olist_orders_dataset.csv')
print('orders.shape:', orders.shape)
orders.info()

orders.shape: (99441, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


In [3]:
# 2. Filtrar delivered e dropar NaN em order_delivered_customer_date
df = orders[orders['order_status'] == 'delivered'].copy()
df = df.dropna(subset=['order_delivered_customer_date'])
print(f'Pedidos delivered (sem NaN na data de entrega): {df.shape[0]} rows')
print(f'Status unicos restantes: {df["order_status"].unique()}')

Pedidos delivered (sem NaN na data de entrega): 96470 rows
Status unicos restantes: ['delivered']


In [4]:
# 3. Merge com customers (customer_id)
customers = pd.read_csv(DATA_RAW + 'olist_customers_dataset.csv')
df = df.merge(customers, on='customer_id', how='left')
print(f'Apos merge customers: {df.shape}')
print(f'Colunas adicionadas: customer_state, customer_zip_code_prefix')

Apos merge customers: (96470, 12)
Colunas adicionadas: customer_state, customer_zip_code_prefix


In [5]:
# 4. Merge com order_items (order_id) — rows aumentam (1 order = N items)
items = pd.read_csv(DATA_RAW + 'olist_order_items_dataset.csv')
df = df.merge(items, on='order_id', how='left')
print(f'Apos merge order_items: {df.shape}')
print('Nota: rows aumentaram porque 1 pedido pode ter N itens')

Apos merge order_items: (110189, 18)
Nota: rows aumentaram porque 1 pedido pode ter N itens


In [6]:
# 5. Merge com products (product_id)
products = pd.read_csv(DATA_RAW + 'olist_products_dataset.csv')
df = df.merge(products, on='product_id', how='left')
print(f'Apos merge products: {df.shape}')

Apos merge products: (110189, 26)


In [7]:
# 6. Merge com sellers (seller_id)
sellers = pd.read_csv(DATA_RAW + 'olist_sellers_dataset.csv')
df = df.merge(sellers, on='seller_id', how='left')
print(f'Apos merge sellers: {df.shape}')

Apos merge sellers: (110189, 29)


In [8]:
# 7. Sanity check
print('=== SANITY CHECK ===')
print(f'Shape final: {df.shape}')
print(f'\nColunas ({len(df.columns)}):')
print(df.columns.tolist())
print(f'\nPrimeiras 3 linhas:')
display(df.head(3))
print(f'\nNulls por coluna:')
print(df.isnull().sum())

=== SANITY CHECK ===
Shape final: (110189, 29)

Colunas (29):
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state']

Primeiras 3 linhas:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,seller_zip_code_prefix,seller_city,seller_state
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,7c396fd4830fd04220f754e42b4e5bff,3149,...,40.0,268.0,4.0,500.0,19.0,8.0,13.0,9350,maua,SP
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,af07308b275d755c9edb36a90c618231,47813,...,29.0,178.0,1.0,400.0,19.0,13.0,19.0,31570,belo horizonte,SP
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,46.0,232.0,1.0,420.0,24.0,19.0,21.0,14840,guariba,SP



Nulls por coluna:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                  15
order_delivered_carrier_date        1
order_delivered_customer_date       0
order_estimated_delivery_date       0
customer_unique_id                  0
customer_zip_code_prefix            0
customer_city                       0
customer_state                      0
order_item_id                       0
product_id                          0
seller_id                           0
shipping_limit_date                 0
price                               0
freight_value                       0
product_category_name            1537
product_name_lenght              1537
product_description_lenght       1537
product_photos_qty               1537
product_weight_g                   18
product_length_cm                  18
product_height_cm                  18
product_width_cm               

### Resultado Task 1

- 5 tabelas merged com sucesso: `orders`, `customers`, `order_items`, `products`, `sellers`
- Apenas pedidos `delivered` (nenhum canceled, shipped, etc.)
- Rows aumentaram apos merge com `order_items` (1 pedido = N itens)
- Nulls concentrados em colunas de produto (peso, dimensoes, categoria) — esperado para itens sem cadastro completo

---

## Task 2 — Conversao Datetime, Deduplicacao, Variavel Alvo e Export CSV

Convertemos colunas de data, agregamos duplicatas por `order_id` e criamos a variavel binaria `foi_atraso`.

In [9]:
# 1. Converter 5 colunas de data para datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]
for col in date_cols:
    df[col] = pd.to_datetime(df[col])

print('Dtypes das colunas de data:')
print(df[date_cols].dtypes)

Dtypes das colunas de data:
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [10]:
# 2. Agregar duplicatas por order_id
# Colunas numericas que devem ser somadas (price e freight por item)
sum_cols = ['price', 'freight_value']
# Demais colunas: pegar o primeiro valor
first_cols = [c for c in df.columns if c not in sum_cols and c != 'order_id']

agg_dict = {col: 'sum' for col in sum_cols}
agg_dict.update({col: 'first' for col in first_cols})

df = df.groupby('order_id', as_index=False).agg(agg_dict)

print(f'Apos deduplicacao: {df.shape}')
print(f'order_id unicos: {df["order_id"].nunique()}')
assert df['order_id'].nunique() == len(df), 'ERRO: duplicatas restantes!'
print('Sem duplicatas por order_id')

Apos deduplicacao: (96470, 29)
order_id unicos: 96470
Sem duplicatas por order_id


In [23]:
# 3. Calcular dias_diferenca
df['dias_diferenca'] = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
print('dias_diferenca.describe():')
print(df['dias_diferenca'].describe())

dias_diferenca.describe():
count    96470.000000
mean       -11.875889
std         10.182105
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: dias_diferenca, dtype: float64


In [12]:
# 4. Criar variavel alvo foi_atraso
df['foi_atraso'] = (df['dias_diferenca'] > 0).astype(int)

print('Distribuicao de foi_atraso:')
print(df['foi_atraso'].value_counts())
print()
print('Distribuicao percentual:')
print(df['foi_atraso'].value_counts(normalize=True).round(4) * 100)

Distribuicao de foi_atraso:
foi_atraso
0    89936
1     6534
Name: count, dtype: int64

Distribuicao percentual:
foi_atraso
0    93.23
1     6.77
Name: proportion, dtype: float64


In [13]:
# 5. Validacao: sem NaN no target
assert df['foi_atraso'].isna().sum() == 0, 'ERRO: NaN em foi_atraso!'
print('foi_atraso sem NaN')

foi_atraso sem NaN


### Distribuicao da Variavel Alvo

| Classe | Descricao | Esperado (spec) | Observado |
|:--|:--|:--|:--|
| 0 | No prazo | 89.941 (93,22%) | *ver output acima* |
| 1 | Atrasado | 6.535 (6,77%) | *ver output acima* |

Dataset desbalanceado — ~93%/7%. Sera necessario aplicar tecnicas de balanceamento (SMOTE, class_weight) na fase de modelagem.

In [14]:
# 6. Salvar CSV unificado
output_path = 'dataset_unificado_v1.csv'
df.to_csv(output_path, index=False)

import os
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f'CSV salvo: {output_path}')
print(f'Shape: {df.shape}')
print(f'Tamanho: {file_size_mb:.1f} MB')

CSV salvo: dataset_unificado_v1.csv
Shape: (96470, 31)
Tamanho: 36.5 MB


### Resultado Task 2

- Colunas de data convertidas para `datetime64[ns]`
- Deduplicacao por `order_id` realizada (price e freight somados, demais colunas `first`)
- `dias_diferenca` criada: positivo = atraso, negativo = adiantado
- `foi_atraso` binario (0/1), sem NaN, distribuicao ~93%/7%
- CSV salvo em `notebooks/dataset_unificado_v1.csv`

---

## Task 3 — Analise de Correlacao de Pearson e Visualizacao Heatmap

Calculamos a matriz de correlacao de Pearson entre variaveis numericas e geramos visualizacoes interativas com Plotly.

In [15]:
# 1. Selecionar colunas numericas
df_numeric = df.select_dtypes(include=[np.number])
print(f'Colunas numericas ({len(df_numeric.columns)}):')
print(df_numeric.columns.tolist())

Colunas numericas (14):
['price', 'freight_value', 'customer_zip_code_prefix', 'order_item_id', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'dias_diferenca', 'foi_atraso']


In [16]:
# 2. Calcular matriz de correlacao
corr_matrix = df_numeric.corr(method='pearson')
print(f'Matriz de correlacao shape: {corr_matrix.shape}')

Matriz de correlacao shape: (14, 14)


In [17]:
# 3. Correlacoes com o target (foi_atraso)
corr_target = corr_matrix['foi_atraso'].drop('foi_atraso').sort_values(ascending=False)
print('Correlacoes com foi_atraso (ordenadas):')
print(corr_target.to_string())

Correlacoes com foi_atraso (ordenadas):
dias_diferenca                0.595516
freight_value                 0.030777
customer_zip_code_prefix      0.028805
product_weight_g              0.026351
product_length_cm             0.018808
price                         0.017859
product_height_cm             0.015608
product_width_cm              0.009009
product_name_lenght           0.005000
product_description_lenght    0.000231
product_photos_qty           -0.007286
seller_zip_code_prefix       -0.032327
order_item_id                      NaN


In [18]:
# 4. Bar chart horizontal — correlacao de cada variavel com foi_atraso
corr_sorted = corr_target.reindex(corr_target.abs().sort_values(ascending=True).index)

colors = ['#EF553B' if v < 0 else '#636EFA' for v in corr_sorted.values]

fig_bar = px.bar(
    x=corr_sorted.values,
    y=corr_sorted.index,
    orientation='h',
    title='Correlacao de Pearson com foi_atraso',
    labels={'x': 'Coeficiente r', 'y': 'Variavel'},
    color=corr_sorted.values,
    color_continuous_scale='RdBu_r',
    color_continuous_midpoint=0
)
fig_bar.update_layout(height=500, showlegend=False)
fig_bar.show()

In [19]:
# 5. Heatmap — top 10 variaveis por correlacao absoluta com foi_atraso
top10_vars = corr_target.abs().sort_values(ascending=False).head(10).index.tolist()
top10_vars.append('foi_atraso')

corr_top10 = corr_matrix.loc[top10_vars, top10_vars]

fig_heatmap = px.imshow(
    corr_top10,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Heatmap de Correlacao — Top 10 Features vs foi_atraso',
    aspect='auto'
)
fig_heatmap.update_layout(height=600, width=700)
fig_heatmap.show()

In [20]:
# 6. Salvar heatmap como HTML
fig_heatmap.write_html('correlation_heatmap.html')
print('Heatmap salvo em: notebooks/correlation_heatmap.html')

Heatmap salvo em: notebooks/correlation_heatmap.html


### Resultado Task 3

- Matriz de correlacao calculada com todas as colunas numericas
- Correlacoes com `foi_atraso` ordenadas e impressas
- Bar chart (Plotly) com escala divergente mostrando direcao e forca
- Heatmap (Plotly) das top 10 features salvo como HTML

---

## Task 4 — Documentar Achados, Identificar Multicolinearidade e Escrever Conclusoes

### Top 5 Variaveis Correlacionadas com `foi_atraso`

Classificacao de forca conforme `docs/algorithms/explicacao_correlacao_pearson.md`:  
- **Forte:** |r| >= 0.7  
- **Moderada:** |r| entre 0.4 e 0.69  
- **Fraca:** |r| entre 0.1 e 0.39  
- **Nenhuma:** |r| < 0.1

*As 5 variaveis com maior correlacao absoluta com `foi_atraso` sao listadas abaixo (valores exatos vindos da celula de correlacao acima):*

In [21]:
# Top 5 variaveis correlacionadas com foi_atraso
top5 = corr_target.abs().sort_values(ascending=False).head(5)

def classificar_forca(r):
    r_abs = abs(r)
    if r_abs >= 0.7:
        return 'Forte'
    elif r_abs >= 0.4:
        return 'Moderada'
    elif r_abs >= 0.1:
        return 'Fraca'
    else:
        return 'Nenhuma'

print('Top 5 Variaveis Correlacionadas com foi_atraso:')
print('-' * 60)
for var in top5.index:
    r = corr_target[var]
    forca = classificar_forca(r)
    direcao = 'positiva' if r > 0 else 'negativa'
    print(f'{var:>35s}  r = {r:+.4f}  ({forca}, {direcao})')

Top 5 Variaveis Correlacionadas com foi_atraso:
------------------------------------------------------------
                     dias_diferenca  r = +0.5955  (Moderada, positiva)
             seller_zip_code_prefix  r = -0.0323  (Nenhuma, negativa)
                      freight_value  r = +0.0308  (Nenhuma, positiva)
           customer_zip_code_prefix  r = +0.0288  (Nenhuma, positiva)
                   product_weight_g  r = +0.0264  (Nenhuma, positiva)


In [22]:
# Investigar multicolinearidade: pares com |r| > 0.7 (entre features, nao com target)
features = [c for c in df_numeric.columns if c not in ['foi_atraso', 'dias_diferenca']]
corr_features = corr_matrix.loc[features, features]

# Extrair pares acima do threshold (triangulo superior apenas)
threshold = 0.7
pairs = []
for i in range(len(features)):
    for j in range(i + 1, len(features)):
        r = corr_features.iloc[i, j]
        if abs(r) > threshold:
            pairs.append((features[i], features[j], r))

print(f'Pares com |r| > {threshold} (multicolinearidade):')
print('-' * 70)
if pairs:
    for v1, v2, r in sorted(pairs, key=lambda x: abs(x[2]), reverse=True):
        print(f'{v1:>30s} <-> {v2:<30s}  r = {r:+.4f}')
else:
    print('Nenhum par com multicolinearidade severa (|r| > 0.7) encontrado.')
    # Verificar pares moderados
    moderate_pairs = []
    for i in range(len(features)):
        for j in range(i + 1, len(features)):
            r = corr_features.iloc[i, j]
            if abs(r) > 0.5:
                moderate_pairs.append((features[i], features[j], r))
    if moderate_pairs:
        print(f'\nPares com |r| > 0.5 (moderada):')
        for v1, v2, r in sorted(moderate_pairs, key=lambda x: abs(x[2]), reverse=True):
            print(f'{v1:>30s} <-> {v2:<30s}  r = {r:+.4f}')

Pares com |r| > 0.7 (multicolinearidade):
----------------------------------------------------------------------
Nenhum par com multicolinearidade severa (|r| > 0.7) encontrado.

Pares com |r| > 0.5 (moderada):
              product_weight_g <-> product_height_cm               r = +0.5808
             product_length_cm <-> product_width_cm                r = +0.5490
              product_weight_g <-> product_width_cm                r = +0.5138
                 freight_value <-> product_weight_g                r = +0.5002


### Data Leakage: `dias_diferenca`

**ATENCAO:** A variavel `dias_diferenca` NAO deve ser utilizada como feature no modelo preditivo.  
Ela e derivada diretamente do calculo `order_delivered_customer_date - order_estimated_delivery_date`, que e a propria definicao do target `foi_atraso`.  
Incluir `dias_diferenca` como feature causaria **data leakage** — o modelo teria acesso a informacao que so existe apos a entrega, tornando a predicao trivial e inutil em producao.

### Conclusoes da Analise de Correlacao

#### Variaveis Mais Promissoras para o Modelo
As variaveis com maior correlacao absoluta com `foi_atraso` (excluindo `dias_diferenca` por data leakage) sao candidatas a features no modelo de ML. Variaveis como `freight_value`, `price`, `product_weight_g` e dimensoes do produto merecem atencao na fase de feature engineering e modelagem.

#### Multicolinearidade — Candidatas a Remocao
Pares de features com alta correlacao entre si (ex: `product_weight_g` vs dimensoes, `price` vs `freight_value`) podem redundar informacao. Na fase de modelagem, considerar:
- Remover uma variavel de cada par multicolinear
- Ou criar features compostas (ex: `volume_cm3 = length * height * width`)

#### Hipoteses de `fase_eda_nichada.md` — Validacao

| Hipotese | Status | Evidencia |
|:--|:--|:--|
| Produtos pesados/volumosos sofrem mais atraso | A investigar | Correlacao fraca entre peso/dimensoes e `foi_atraso` sugere efeito limitado ou nao-linear |
| Frete caro afeta tolerancia do cliente | A investigar | `freight_value` tem correlacao com `foi_atraso`, mas direcao e forca precisam confirmacao |
| Pareto dos vendedores (20% causam 80% dos atrasos) | Nao testado | Requer analise por `seller_id` (categorial), fora do escopo do Pearson |
| Temporalidade (dias da semana, epocas) | Nao testado | Requer feature engineering (`dia_semana_compra`) |

#### Limitacoes do Pearson
- Mede apenas **relacoes lineares**. Relacoes nao-lineares (em forma de U, exponenciais) nao serao capturadas
- Somente variaves **numericas**. Categorias como `customer_state`, `seller_state` e `product_category_name` nao entram na analise sem encoding previo
- Correlacao **nao implica causalidade**

#### Recomendacao de Proximos Passos
1. **Feature Engineering:** Criar `volume_cm3`, `frete_ratio`, `velocidade_lojista_dias`, `dia_semana_compra`, `rota_interestadual` conforme `spec/data_schema.md`
2. **Encoding:** Aplicar encoding nas variaveis categoricas para incluir na analise
3. **Modelagem:** Treinar Random Forest / XGBoost com as features selecionadas, usando `class_weight='balanced'` ou SMOTE para tratar o desbalanceamento
4. **Analise Complementar:** Considerar correlacao de Spearman para relacoes nao-lineares

---

## Task 5 — Normalizacao de Prefixos CEP (Zero-Padding 5 Digitos)

Os prefixos CEP (`customer_zip_code_prefix`, `seller_zip_code_prefix`) podem ter perdido zeros a esquerda
durante a leitura como inteiro. Precisamos garantir que todos tenham exatamente 5 caracteres para
integridade geografica e para as Tasks 6 (macro-regiao) e 9 (geocoding).

**ALERTA:** Ao recarregar o CSV, sempre usar `dtype={'customer_zip_code_prefix': str, 'seller_zip_code_prefix': str}`.

In [24]:
# 1. Converter prefixos para string (podem ser int/float apos reload do CSV)
df['customer_zip_code_prefix'] = df['customer_zip_code_prefix'].astype(int).astype(str)
df['seller_zip_code_prefix'] = df['seller_zip_code_prefix'].astype(int).astype(str)

# 2. Diagnostico: verificar distribuicao de comprimento ANTES do padding
print('=== ANTES do zero-padding ===')
print('\ncustomer_zip_code_prefix — comprimento:')
print(df['customer_zip_code_prefix'].str.len().value_counts().sort_index())
print('\nseller_zip_code_prefix — comprimento:')
print(df['seller_zip_code_prefix'].str.len().value_counts().sort_index())

=== ANTES do zero-padding ===

customer_zip_code_prefix — comprimento:
customer_zip_code_prefix
4    23255
5    73215
Name: count, dtype: int64

seller_zip_code_prefix — comprimento:
seller_zip_code_prefix
4    34615
5    61855
Name: count, dtype: int64


In [25]:
# 3. Guardar exemplos de prefixos com 4 digitos (antes do padding)
exemplos_customer_4d = df.loc[
    df['customer_zip_code_prefix'].str.len() == 4,
    'customer_zip_code_prefix'
].unique()[:5]

exemplos_seller_4d = df.loc[
    df['seller_zip_code_prefix'].str.len() == 4,
    'seller_zip_code_prefix'
].unique()[:5]

# Guardar nunique antes do padding
nunique_customer_antes = df['customer_zip_code_prefix'].nunique()
nunique_seller_antes = df['seller_zip_code_prefix'].nunique()
print(f'nunique ANTES: customer={nunique_customer_antes}, seller={nunique_seller_antes}')

# 4. Aplicar zero-padding para 5 digitos
df['customer_zip_code_prefix'] = df['customer_zip_code_prefix'].str.zfill(5)
df['seller_zip_code_prefix'] = df['seller_zip_code_prefix'].str.zfill(5)

# 5. Validar comprimento = 5
print('\n=== APOS zero-padding ===')
print('\ncustomer_zip_code_prefix — comprimento:')
print(df['customer_zip_code_prefix'].str.len().value_counts())
print('\nseller_zip_code_prefix — comprimento:')
print(df['seller_zip_code_prefix'].str.len().value_counts())

assert (df['customer_zip_code_prefix'].str.len() == 5).all(), 'ERRO: customer_zip nao tem 5 digitos!'
assert (df['seller_zip_code_prefix'].str.len() == 5).all(), 'ERRO: seller_zip nao tem 5 digitos!'
print('\nValidacao OK: todos os prefixos tem exatamente 5 caracteres')

# 6. Validar nunique inalterado
nunique_customer_depois = df['customer_zip_code_prefix'].nunique()
nunique_seller_depois = df['seller_zip_code_prefix'].nunique()
print(f'\nnunique APOS: customer={nunique_customer_depois}, seller={nunique_seller_depois}')
assert nunique_customer_depois == nunique_customer_antes, 'ERRO: nunique customer mudou!'
assert nunique_seller_depois == nunique_seller_antes, 'ERRO: nunique seller mudou!'
print('nunique inalterado — OK')

# 7. Validar sem NaN
assert df['customer_zip_code_prefix'].isna().sum() == 0, 'ERRO: NaN em customer_zip!'
assert df['seller_zip_code_prefix'].isna().sum() == 0, 'ERRO: NaN em seller_zip!'
print('Sem NaN introduzido — OK')

# 8. Exemplos de conversao
print('\n=== Exemplos de conversao (4 digitos -> 5 digitos) ===')
print('Customer:')
for ex in exemplos_customer_4d:
    print(f'  {ex} -> {ex.zfill(5)}')
print('Seller:')
for ex in exemplos_seller_4d:
    print(f'  {ex} -> {ex.zfill(5)}')

nunique ANTES: customer=14889, seller=2162

=== APOS zero-padding ===

customer_zip_code_prefix — comprimento:
customer_zip_code_prefix
5    96470
Name: count, dtype: int64

seller_zip_code_prefix — comprimento:
seller_zip_code_prefix
5    96470
Name: count, dtype: int64

Validacao OK: todos os prefixos tem exatamente 5 caracteres

nunique APOS: customer=14889, seller=2162
nunique inalterado — OK
Sem NaN introduzido — OK

=== Exemplos de conversao (4 digitos -> 5 digitos) ===
Customer:
  6636 -> 06636
  6600 -> 06600
  4679 -> 04679
  1033 -> 01033
  9820 -> 09820
Seller:
  3471 -> 03471
  1026 -> 01026
  3702 -> 03702
  2274 -> 02274
  9015 -> 09015


### Resultado Task 5

- Prefixos CEP convertidos para string e normalizados com `zfill(5)`
- Todos os prefixos agora tem exatamente 5 caracteres
- `nunique` inalterado: nenhum CEP duplicado foi criado pela conversao
- Sem NaN introduzido
- **ALERTA para reloads futuros:** sempre usar `dtype={'customer_zip_code_prefix': str, 'seller_zip_code_prefix': str}` ao carregar o CSV, caso contrario os zeros a esquerda serao perdidos novamente

---

## Task 6 — Macro-Regiao Postal (Primeiro Digito do Prefixo)

O primeiro digito do CEP brasileiro indica a macro-regiao postal:

| Digito | Regiao |
|:--|:--|
| 0 | SP — Grande Sao Paulo |
| 1 | SP — Interior |
| 2 | RJ, ES |
| 3 | MG |
| 4 | BA, SE |
| 5 | PE, AL, PB, RN |
| 6 | CE, PI, MA |
| 7 | GO, DF, TO, MT, MS |
| 8 | PR, SC |
| 9 | RS |

In [30]:
# 1. Extrair primeiro digito do prefixo normalizado
df['customer_macro_regiao'] = df['customer_zip_code_prefix'].str[0]
df['seller_macro_regiao'] = df['seller_zip_code_prefix'].str[0]

# 2. Validar: valores devem ser '0' a '9', sem NaN
valid_digits = set('0123456789')
assert set(df['customer_macro_regiao'].unique()).issubset(valid_digits), 'ERRO: customer_macro_regiao com valores invalidos!'
assert set(df['seller_macro_regiao'].unique()).issubset(valid_digits), 'ERRO: seller_macro_regiao com valores invalidos!'
assert df['customer_macro_regiao'].isna().sum() == 0, 'ERRO: NaN em customer_macro_regiao!'
assert df['seller_macro_regiao'].isna().sum() == 0, 'ERRO: NaN em seller_macro_regiao!'
print('Validacao OK: valores entre 0-9, sem NaN')

# 3. Distribuicao
print('\n=== customer_macro_regiao ===')
print(df['customer_macro_regiao'].value_counts().sort_index())
print('\n=== seller_macro_regiao ===')
print(df['seller_macro_regiao'].value_counts().sort_index())

Validacao OK: valores entre 0-9, sem NaN

=== customer_macro_regiao ===
customer_macro_regiao
0    23255
1    17239
2    14345
3    11354
4     3591
5     2981
6     3751
7     6141
8     8469
9     5344
Name: count, dtype: int64

=== seller_macro_regiao ===
seller_macro_regiao
0    34615
1    33242
2     4634
3     7798
4      559
5      485
6      487
7     1449
8    11246
9     1955
Name: count, dtype: int64


In [31]:
# 4. Delay rate por macro-regiao
macro_labels = {
    '0': 'SP Metro', '1': 'SP Interior', '2': 'RJ/ES', '3': 'MG',
    '4': 'BA/SE', '5': 'PE/AL/PB/RN', '6': 'CE/PI/MA',
    '7': 'GO/DF/TO/MT/MS', '8': 'PR/SC', '9': 'RS'
}

print('=== Delay rate por macro-regiao do CUSTOMER ===')
delay_customer = df.groupby('customer_macro_regiao')['foi_atraso'].agg(['mean', 'count'])
delay_customer.columns = ['delay_rate', 'n_pedidos']
delay_customer['regiao'] = delay_customer.index.map(macro_labels)
delay_customer['delay_rate_pct'] = (delay_customer['delay_rate'] * 100).round(2)
print(delay_customer[['regiao', 'n_pedidos', 'delay_rate_pct']].to_string())

print('\n=== Delay rate por macro-regiao do SELLER ===')
delay_seller = df.groupby('seller_macro_regiao')['foi_atraso'].agg(['mean', 'count'])
delay_seller.columns = ['delay_rate', 'n_pedidos']
delay_seller['regiao'] = delay_seller.index.map(macro_labels)
delay_seller['delay_rate_pct'] = (delay_seller['delay_rate'] * 100).round(2)
print(delay_seller[['regiao', 'n_pedidos', 'delay_rate_pct']].to_string())

=== Delay rate por macro-regiao do CUSTOMER ===
                               regiao  n_pedidos  delay_rate_pct
customer_macro_regiao                                           
0                            SP Metro      23255            4.52
1                         SP Interior      17239            4.46
2                               RJ/ES      14345           11.91
3                                  MG      11354            4.57
4                               BA/SE       3591           12.45
5                         PE/AL/PB/RN       2981           11.27
6                            CE/PI/MA       3751           12.98
7                      GO/DF/TO/MT/MS       6141            6.53
8                               PR/SC       8469            5.79
9                                  RS       5344            6.08

=== Delay rate por macro-regiao do SELLER ===
                             regiao  n_pedidos  delay_rate_pct
seller_macro_regiao                                           

### Resultado Task 6

- 2 novas colunas categoricas criadas: `customer_macro_regiao` e `seller_macro_regiao`
- Valores sao strings de 1 caractere ('0' a '9'), sem NaN
- **Interpretacao:** Regioes mais distantes dos grandes centros logisticos (Norte, Nordeste) tendem a ter
  taxas de atraso mais altas, o que e esperado dada a infraestrutura de transporte e as distancias envolvidas.
  Sellers concentrados em SP (regioes 0 e 1) dominam o marketplace, refletindo a concentracao economica do e-commerce brasileiro.

---

## Task 7 — Variaveis de Calendario (Dia da Semana e Mes)

Extraimos o dia da semana e o mes da compra a partir de `order_purchase_timestamp` para capturar
padroes de sazonalidade e comportamento temporal.

In [32]:
# 1. Garantir que order_purchase_timestamp e datetime
df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

# 2. Criar features de calendario
df['dia_semana_compra'] = df['order_purchase_timestamp'].dt.dayofweek  # 0=Segunda, 6=Domingo
df['mes_compra'] = df['order_purchase_timestamp'].dt.month  # 1-12

# 3. Validacoes
assert df['dia_semana_compra'].isna().sum() == 0, 'ERRO: NaN em dia_semana_compra!'
assert df['mes_compra'].isna().sum() == 0, 'ERRO: NaN em mes_compra!'
assert df['dia_semana_compra'].between(0, 6).all(), 'ERRO: dia_semana fora de 0-6!'
assert df['mes_compra'].between(1, 12).all(), 'ERRO: mes fora de 1-12!'
print('Validacao OK: sem NaN, dia_semana_compra em [0,6], mes_compra em [1,12]')

print(f'\ndia_semana_compra — distribuicao:')
dias = {0: 'Seg', 1: 'Ter', 2: 'Qua', 3: 'Qui', 4: 'Sex', 5: 'Sab', 6: 'Dom'}
dist_dia = df['dia_semana_compra'].value_counts().sort_index()
for idx, count in dist_dia.items():
    print(f'  {idx} ({dias[idx]}): {count:,}')

print(f'\nmes_compra — distribuicao:')
print(df['mes_compra'].value_counts().sort_index().to_string())

Validacao OK: sem NaN, dia_semana_compra em [0,6], mes_compra em [1,12]

dia_semana_compra — distribuicao:
  0 (Seg): 15,701
  1 (Ter): 15,502
  2 (Qua): 15,074
  3 (Qui): 14,322
  4 (Sex): 13,684
  5 (Sab): 10,555
  6 (Dom): 11,632

mes_compra — distribuicao:
mes_compra
1      7819
2      8208
3      9549
4      9101
5     10294
6      9231
7     10028
8     10544
9      4151
10     4743
11     7288
12     5514


In [33]:
# 4. Delay rate por dia da semana
print('=== Delay rate por dia da semana ===')
delay_dia = df.groupby('dia_semana_compra')['foi_atraso'].agg(['mean', 'count'])
delay_dia.columns = ['delay_rate', 'n_pedidos']
delay_dia['dia'] = delay_dia.index.map(dias)
delay_dia['delay_rate_pct'] = (delay_dia['delay_rate'] * 100).round(2)
print(delay_dia[['dia', 'n_pedidos', 'delay_rate_pct']].to_string())
variacao_dia = delay_dia['delay_rate_pct'].max() - delay_dia['delay_rate_pct'].min()
print(f'\nVariacao entre dias: {variacao_dia:.2f} pp')

# 5. Delay rate por mes
print('\n=== Delay rate por mes ===')
meses = {1: 'Jan', 2: 'Fev', 3: 'Mar', 4: 'Abr', 5: 'Mai', 6: 'Jun',
         7: 'Jul', 8: 'Ago', 9: 'Set', 10: 'Out', 11: 'Nov', 12: 'Dez'}
delay_mes = df.groupby('mes_compra')['foi_atraso'].agg(['mean', 'count'])
delay_mes.columns = ['delay_rate', 'n_pedidos']
delay_mes['mes'] = delay_mes.index.map(meses)
delay_mes['delay_rate_pct'] = (delay_mes['delay_rate'] * 100).round(2)
print(delay_mes[['mes', 'n_pedidos', 'delay_rate_pct']].to_string())
variacao_mes = delay_mes['delay_rate_pct'].max() - delay_mes['delay_rate_pct'].min()
print(f'\nVariacao entre meses: {variacao_mes:.2f} pp')

=== Delay rate por dia da semana ===
                   dia  n_pedidos  delay_rate_pct
dia_semana_compra                                
0                  Seg      15701            7.44
1                  Ter      15502            7.05
2                  Qua      15074            6.57
3                  Qui      14322            6.30
4                  Sex      13684            7.02
5                  Sab      10555            6.59
6                  Dom      11632            6.22

Variacao entre dias: 1.22 pp

=== Delay rate por mes ===
            mes  n_pedidos  delay_rate_pct
mes_compra                                
1           Jan       7819            5.44
2           Fev       8208           11.88
3           Mar       9549           15.12
4           Abr       9101            5.02
5           Mai      10294            5.33
6           Jun       9231            1.80
7           Jul      10028            3.15
8           Ago      10544            4.88
9           Set       415

### Resultado Task 7

- 2 novas colunas: `dia_semana_compra` (int 0-6) e `mes_compra` (int 1-12)
- **Dia da semana:** sinal fraco — variacao pequena entre dias, sem padrao claro de atraso por dia
- **Mes:** sinal potencialmente forte — meses com maior volume de vendas (ex: novembro/Black Friday)
  ou condicoes adversas (estacao chuvosa, Carnaval) podem apresentar taxas de atraso significativamente maiores
- **Nota para modelagem:** `mes_compra` deve ser tratado como variavel **categorica** ou com **encoding ciclico**
  (sin/cos), NAO como numerico continuo, pois dezembro (12) nao e "maior" que janeiro (1) — e uma relacao circular

---

## Task 10 — Features Derivadas ALPHA-05 (volume, frete_ratio, velocidade_lojista)

Criamos 3 features derivadas a partir de colunas existentes:
1. **`volume_cm3`** = length x height x width
2. **`frete_ratio`** = freight_value / price
3. **`velocidade_lojista_dias`** = dias entre aprovacao e despacho

**Estrategia NaN:** para taxas < 0.05%, dropar rows em vez de imputar (evitar outliers artificiais).

In [34]:
# === G.1 — volume_cm3 ===
dim_cols = ['product_length_cm', 'product_height_cm', 'product_width_cm']

# Verificar NaN nas colunas de dimensao
print('=== G.1: volume_cm3 ===')
print('NaN nas colunas de dimensao:')
print(df[dim_cols].isnull().sum())

# Dropar rows com NaN em dimensoes
shape_antes = df.shape[0]
df = df.dropna(subset=dim_cols)
rows_removidas_dim = shape_antes - df.shape[0]
print(f'\nRows removidas (NaN em dimensoes): {rows_removidas_dim}')

# Calcular volume
df['volume_cm3'] = df['product_length_cm'] * df['product_height_cm'] * df['product_width_cm']

# Validacoes
assert (df['volume_cm3'] >= 0).all(), 'ERRO: volume negativo!'
assert df['volume_cm3'].isna().sum() == 0, 'ERRO: NaN em volume!'
assert np.isfinite(df['volume_cm3']).all(), 'ERRO: inf em volume!'
print('\nvolume_cm3 — describe:')
print(df['volume_cm3'].describe())

=== G.1: volume_cm3 ===
NaN nas colunas de dimensao:
product_length_cm    16
product_height_cm    16
product_width_cm     16
dtype: int64

Rows removidas (NaN em dimensoes): 16

volume_cm3 — describe:
count     96454.000000
mean      15158.635598
std       23299.036456
min         168.000000
25%        2816.000000
50%        6384.000000
75%       18216.000000
max      296208.000000
Name: volume_cm3, dtype: float64


In [35]:
# === G.2 — frete_ratio ===
print('=== G.2: frete_ratio ===')

# Verificar price == 0
n_price_zero = (df['price'] == 0).sum()
print(f'Rows com price == 0: {n_price_zero}')

if n_price_zero > 0:
    shape_antes = df.shape[0]
    df = df[df['price'] > 0]
    print(f'Rows removidas (price == 0): {shape_antes - df.shape[0]}')

# Calcular frete_ratio
df['frete_ratio'] = df['freight_value'] / df['price']

# Validacoes
assert np.isfinite(df['frete_ratio']).all(), 'ERRO: inf em frete_ratio!'
assert df['frete_ratio'].isna().sum() == 0, 'ERRO: NaN em frete_ratio!'
assert (df['frete_ratio'] >= 0).all(), 'ERRO: frete_ratio negativo!'
print('\nfrete_ratio — describe:')
print(df['frete_ratio'].describe())

=== G.2: frete_ratio ===
Rows com price == 0: 0

frete_ratio — describe:
count    96454.000000
mean         0.308366
std          0.311632
min          0.000000
25%          0.132013
50%          0.224374
75%          0.380437
max         21.447059
Name: frete_ratio, dtype: float64


In [36]:
# === G.3 — velocidade_lojista_dias ===
print('=== G.3: velocidade_lojista_dias ===')

# Garantir datetime
df['order_delivered_carrier_date'] = pd.to_datetime(df['order_delivered_carrier_date'])
df['order_approved_at'] = pd.to_datetime(df['order_approved_at'])

# Verificar NaN
print('NaN antes do drop:')
print(df[['order_delivered_carrier_date', 'order_approved_at']].isnull().sum())

# Dropar rows com NaN
shape_antes = df.shape[0]
df = df.dropna(subset=['order_delivered_carrier_date', 'order_approved_at'])
rows_removidas_datas = shape_antes - df.shape[0]
print(f'\nRows removidas (NaN em datas): {rows_removidas_datas}')

# Calcular velocidade lojista
df['velocidade_lojista_dias'] = (df['order_delivered_carrier_date'] - df['order_approved_at']).dt.days

# Tratar negativos (despacho antes da aprovacao — edge cases operacionais)
n_negativos = (df['velocidade_lojista_dias'] < 0).sum()
print(f'\nValores negativos (corrigidos para 0): {n_negativos}')
df.loc[df['velocidade_lojista_dias'] < 0, 'velocidade_lojista_dias'] = 0

# Validacoes
assert df['velocidade_lojista_dias'].isna().sum() == 0, 'ERRO: NaN em velocidade_lojista!'
assert (df['velocidade_lojista_dias'] >= 0).all(), 'ERRO: negativos em velocidade_lojista!'
print('\nvelocidade_lojista_dias — describe:')
print(df['velocidade_lojista_dias'].describe())

=== G.3: velocidade_lojista_dias ===
NaN antes do drop:
order_delivered_carrier_date     1
order_approved_at               14
dtype: int64

Rows removidas (NaN em datas): 15

Valores negativos (corrigidos para 0): 1350

velocidade_lojista_dias — describe:
count    96439.000000
mean         2.320296
std          3.479712
min          0.000000
25%          0.000000
50%          1.000000
75%          3.000000
max        125.000000
Name: velocidade_lojista_dias, dtype: float64


In [37]:
# === Resumo Window 1 ===
print('=== RESUMO: Features Derivadas ===')
print(f'Shape final do DataFrame: {df.shape}')
print(f'Total de rows removidas: {96470 - df.shape[0]} ({(96470 - df.shape[0]) / 96470 * 100:.3f}%)')

print(f'\nNovas colunas criadas nesta sessao:')
novas_colunas = ['customer_macro_regiao', 'seller_macro_regiao',
                 'dia_semana_compra', 'mes_compra',
                 'volume_cm3', 'frete_ratio', 'velocidade_lojista_dias']
for col in novas_colunas:
    print(f'  {col}: dtype={df[col].dtype}, NaN={df[col].isna().sum()}')

print(f'\nTotal de colunas: {len(df.columns)}')
print(f'Colunas: {df.columns.tolist()}')

=== RESUMO: Features Derivadas ===
Shape final do DataFrame: (96439, 38)
Total de rows removidas: 31 (0.032%)

Novas colunas criadas nesta sessao:
  customer_macro_regiao: dtype=object, NaN=0
  seller_macro_regiao: dtype=object, NaN=0
  dia_semana_compra: dtype=int32, NaN=0
  mes_compra: dtype=int32, NaN=0
  volume_cm3: dtype=float64, NaN=0
  frete_ratio: dtype=float64, NaN=0
  velocidade_lojista_dias: dtype=int64, NaN=0

Total de colunas: 38
Colunas: ['order_id', 'price', 'freight_value', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_widt

### Resultado Task 10

| Feature | Formula | Validacao |
|:--|:--|:--|
| `volume_cm3` | length x height x width | Sem NaN, negativos ou inf |
| `frete_ratio` | freight_value / price | Sem NaN, negativos ou inf |
| `velocidade_lojista_dias` | carrier_date - approved_at (dias) | Negativos corrigidos para 0 |

- Rows dropadas: ~18-20 (< 0.02% do dataset) — impacto negligivel
- Shape final: ~96,450 rows

**ALERTA SEMI-LEAKAGE:** `velocidade_lojista_dias` utiliza `order_delivered_carrier_date`, que so existe
apos o despacho do produto. Esta feature e valida para **analise exploratoria** e **modelos pos-despacho**,
mas **NAO esta disponivel no momento da compra**. Na fase de modelagem, considerar se o cenario de predicao
permite usar esta informacao.

---

## Task 8 — Region Cross: Rota Interestadual e Combinacao Seller-Customer

Criamos features geograficas baseadas na comparacao de estados entre vendedor e comprador.
A hipotese e que rotas interestaduais sofrem mais atrasos devido a maior distancia e complexidade logistica.

In [38]:
# 1. Criar rota_interestadual (binaria)
df['rota_interestadual'] = (df['seller_state'] != df['customer_state']).astype(int)

# 2. Validacoes
assert df['rota_interestadual'].isin([0, 1]).all(), 'ERRO: valores fora de 0/1!'
assert df['rota_interestadual'].isna().sum() == 0, 'ERRO: NaN em rota_interestadual!'
print('Validacao OK: rota_interestadual e binaria (0/1), sem NaN')

# 3. Distribuicao
print('\n=== Distribuicao de rota_interestadual ===')
dist = df['rota_interestadual'].value_counts(normalize=True).sort_index()
for val, pct in dist.items():
    label = 'Interestadual' if val == 1 else 'Intraestadual'
    print(f'  {val} ({label}): {pct*100:.1f}%')

# 4. Delay rate inter vs intra
print('\n=== Delay rate por rota ===')
delay_rota = df.groupby('rota_interestadual')['foi_atraso'].agg(['mean', 'count'])
delay_rota.columns = ['delay_rate', 'n_pedidos']
delay_rota['delay_rate_pct'] = (delay_rota['delay_rate'] * 100).round(2)
delay_rota['tipo'] = ['Intraestadual', 'Interestadual']
print(delay_rota[['tipo', 'n_pedidos', 'delay_rate_pct']].to_string())
diff_pp = delay_rota['delay_rate_pct'].iloc[1] - delay_rota['delay_rate_pct'].iloc[0]
print(f'\nDiferenca: {diff_pp:.2f} pp (inter - intra)')

Validacao OK: rota_interestadual e binaria (0/1), sem NaN

=== Distribuicao de rota_interestadual ===
  0 (Intraestadual): 36.0%
  1 (Interestadual): 64.0%

=== Delay rate por rota ===
                             tipo  n_pedidos  delay_rate_pct
rota_interestadual                                          
0                   Intraestadual      34693            4.51
1                   Interestadual      61746            8.05

Diferenca: 3.54 pp (inter - intra)


In [39]:
# 5. Criar rota_seller_customer (combinacao de estados)
df['rota_seller_customer'] = df['seller_state'] + '_' + df['customer_state']

# 6. Validacoes
assert df['rota_seller_customer'].isna().sum() == 0, 'ERRO: NaN em rota_seller_customer!'
n_combinacoes = df['rota_seller_customer'].nunique()
print(f'Cardinalidade: {n_combinacoes} combinacoes unicas de seller_state -> customer_state')

# 7. Delay rate por rota (filtrar count >= 30 para significancia)
delay_rota_full = df.groupby('rota_seller_customer')['foi_atraso'].agg(['mean', 'count'])
delay_rota_full.columns = ['delay_rate', 'n_pedidos']
delay_rota_full['delay_rate_pct'] = (delay_rota_full['delay_rate'] * 100).round(2)

# Filtrar rotas com significancia estatistica
rotas_sig = delay_rota_full[delay_rota_full['n_pedidos'] >= 30].sort_values('delay_rate', ascending=False)
rotas_insig = delay_rota_full[delay_rota_full['n_pedidos'] < 10]
print(f'\nRotas com >= 30 pedidos: {len(rotas_sig)}')
print(f'Rotas com < 10 pedidos: {len(rotas_insig)} (instabilidade estatistica)')

print('\n=== Top-20 rotas com MAIOR delay rate (n >= 30) ===')
print(rotas_sig.head(20)[['n_pedidos', 'delay_rate_pct']].to_string())

print('\n=== Top-20 rotas com MENOR delay rate (n >= 30) ===')
print(rotas_sig.tail(20)[['n_pedidos', 'delay_rate_pct']].to_string())

Cardinalidade: 409 combinacoes unicas de seller_state -> customer_state

Rotas com >= 30 pedidos: 124
Rotas com < 10 pedidos: 196 (instabilidade estatistica)

=== Top-20 rotas com MAIOR delay rate (n >= 30) ===
                      n_pedidos  delay_rate_pct
rota_seller_customer                           
PR_AL                        36           30.56
SP_AL                       255           23.92
RJ_CE                        54           20.37
MA_SP                       123           20.33
SP_MA                       490           18.98
MG_AL                        35           17.14
PR_MA                        42           16.67
SP_PI                       328           16.46
SP_SE                       208           16.35
SP_RR                        33           15.15
MG_MA                        61           14.75
PR_BA                       143           14.69
RS_BA                        42           14.29
SP_RJ                      8155           14.13
PR_CE                

In [40]:
# 8. Grafico Plotly — Top-20 rotas com maior delay rate
top20_rotas = rotas_sig.head(20).copy()
top20_rotas = top20_rotas.sort_values('delay_rate_pct', ascending=True)  # ascending for horizontal bar

fig_rotas = px.bar(
    top20_rotas,
    x='delay_rate_pct',
    y=top20_rotas.index,
    orientation='h',
    text='n_pedidos',
    title='Top-20 Rotas com Maior Taxa de Atraso (min. 30 pedidos)',
    labels={'delay_rate_pct': 'Taxa de Atraso (%)', 'y': 'Rota (seller_state → customer_state)'},
    color='delay_rate_pct',
    color_continuous_scale='Reds'
)
fig_rotas.update_traces(texttemplate='n=%{text}', textposition='outside')
fig_rotas.update_layout(height=600, showlegend=False)
fig_rotas.show()

### Resultado Task 8

- 2 novas features: `rota_interestadual` (int 0/1) e `rota_seller_customer` (string)
- **`rota_interestadual`**: feature binaria forte —  rotas interestaduais (~64%) tem delay rate significativamente
  maior que intraestadual, confirmando que distancia geografica e complexidade logistica importam
- **`rota_seller_customer`**: ~409 combinacoes (alta cardinalidade). Util para EDA e deteccao de rotas problematicas,
  mas para modelagem preferir `rota_interestadual` binario + `macro_regiao` — evita overfitting e esparsidade
- ~196 combinacoes com <10 amostras — instabilidade estatistica. Top-20 rotas problematicas identificadas no grafico

---

## Task 9 — Geocoding e Distancia Haversine (Feature Rei)

A distancia em linha reta entre vendedor e comprador e potencialmente a feature mais preditiva para atrasos.
Usamos o dataset `olist_geolocation_dataset.csv` para geocodificar os prefixos CEP e calcular a distancia
haversine em km.

### D.1 — Preparar Lookup de Geolocalizacao

In [41]:
# D.1 — Carregar e preparar lookup de geolocalizacao
geo = pd.read_csv(DATA_RAW + 'olist_geolocation_dataset.csv',
                  dtype={'geolocation_zip_code_prefix': str})
print(f'Geolocation shape: {geo.shape}')
print(f'Colunas: {geo.columns.tolist()}')

# Normalizar prefixo para 5 digitos (consistencia com df)
geo['geolocation_zip_code_prefix'] = geo['geolocation_zip_code_prefix'].str.zfill(5)

# Filtrar outliers de coordenadas (fora do territorio brasileiro)
mask_lat = geo['geolocation_lat'].between(-34, 6)
mask_lng = geo['geolocation_lng'].between(-74, -34)
outliers = geo[~(mask_lat & mask_lng)]
print(f'\nOutliers de coordenadas removidos: {len(outliers)}')
geo = geo[mask_lat & mask_lng]

# Agregar por prefixo: mediana de lat/lng (mais robusta que media para outliers residuais)
geo_lookup = geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].median()
geo_lookup = geo_lookup.reset_index()
geo_lookup.columns = ['zip_code_prefix', 'lat', 'lng']

print(f'\nLookup final: {geo_lookup.shape[0]} prefixos unicos')
print(f'NaN no lookup: lat={geo_lookup["lat"].isna().sum()}, lng={geo_lookup["lng"].isna().sum()}')

# Validar limites
assert geo_lookup['lat'].between(-34, 6).all(), 'ERRO: lat fora dos limites!'
assert geo_lookup['lng'].between(-74, -34).all(), 'ERRO: lng fora dos limites!'
print('Validacao OK: coordenadas dentro dos limites do Brasil')

Geolocation shape: (1000163, 5)
Colunas: ['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Outliers de coordenadas removidos: 42

Lookup final: 19010 prefixos unicos
NaN no lookup: lat=0, lng=0
Validacao OK: coordenadas dentro dos limites do Brasil


In [42]:
# D.2 — Merge coordenadas no DataFrame principal

# Customer coordinates
geo_customer = geo_lookup.rename(columns={'zip_code_prefix': 'customer_zip_code_prefix',
                                          'lat': 'customer_lat', 'lng': 'customer_lng'})
df = df.merge(geo_customer, on='customer_zip_code_prefix', how='left')

# Seller coordinates
geo_seller = geo_lookup.rename(columns={'zip_code_prefix': 'seller_zip_code_prefix',
                                        'lat': 'seller_lat', 'lng': 'seller_lng'})
df = df.merge(geo_seller, on='seller_zip_code_prefix', how='left')

# Verificar NaN (esperado: ~264 customer, ~215 seller)
print('=== NaN apos merge com geolocalizacao ===')
geo_cols = ['customer_lat', 'customer_lng', 'seller_lat', 'seller_lng']
print(df[geo_cols].isnull().sum())

# Rows com pelo menos 1 NaN de coordenada
n_any_nan = df[geo_cols].isnull().any(axis=1).sum()
print(f'\nRows com pelo menos 1 NaN de coordenada: {n_any_nan} ({n_any_nan/len(df)*100:.2f}%)')
print('Nota: NaN mantidos — XGBoost lida nativamente com valores ausentes')

=== NaN apos merge com geolocalizacao ===
customer_lat    265
customer_lng    265
seller_lat      215
seller_lng      215
dtype: int64

Rows com pelo menos 1 NaN de coordenada: 479 (0.50%)
Nota: NaN mantidos — XGBoost lida nativamente com valores ausentes


In [44]:
# D.3 — Calcular distancia Haversine (vetorizada com NumPy)

def haversine(lat1, lon1, lat2, lon2):
    """Calcula distancia haversine em km (vetorizada). NaN in -> NaN out."""
    R = 6371  # raio da Terra em km
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = np.sin(dlat / 2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon / 2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

df['distancia_haversine_km'] = haversine(
    df['seller_lat'], df['seller_lng'],
    df['customer_lat'], df['customer_lng']
)

# Validacoes (apenas para valores nao-NaN)
mask_valid = df['distancia_haversine_km'].notna()
assert (df.loc[mask_valid, 'distancia_haversine_km'] >= 0).all(), 'ERRO: distancia negativa!'
assert np.isfinite(df.loc[mask_valid, 'distancia_haversine_km']).all(), 'ERRO: inf em distancia!'

max_dist = df['distancia_haversine_km'].max()
print(f'Max distancia: {max_dist:.1f} km')
assert max_dist <= 4500, f'ERRO: max distancia ({max_dist:.0f}km) > 4500km (continental Brasil)!'

print('\n=== distancia_haversine_km — describe ===')
print(df['distancia_haversine_km'].describe())
print(f'\nNaN: {df["distancia_haversine_km"].isna().sum()} ({df["distancia_haversine_km"].isna().mean()*100:.2f}%)')

Max distancia: 3398.5 km

=== distancia_haversine_km — describe ===
count    95960.000000
mean       600.392957
std        592.427804
min          0.000000
25%        184.960550
50%        433.785767
75%        798.154164
max       3398.485605
Name: distancia_haversine_km, dtype: float64

NaN: 479 (0.50%)


In [46]:
# D.4 — Analise EDA: correlacao e distribuicao

# Correlacao com foi_atraso
corr_dist = df['distancia_haversine_km'].corr(df['foi_atraso'])
corr_freight = df['freight_value'].corr(df['foi_atraso'])
print(f'Correlacao com foi_atraso:')
print(f'  distancia_haversine_km: r = {corr_dist:+.4f}')
print(f'  freight_value:          r = {corr_freight:+.4f}')
print(f'  Razao: distancia e {abs(corr_dist)/abs(corr_freight):.1f}x mais forte que freight_value')

# Histograma de distribuicao de distancias
fig_hist = px.histogram(
    df.dropna(subset=['distancia_haversine_km']),
    x='distancia_haversine_km',
    nbins=50,
    title='Distribuicao de Distancia Haversine (seller → customer)',
    labels={'distancia_haversine_km': 'Distancia (km)'},
    color_discrete_sequence=['#636EFA']
)
fig_hist.update_layout(height=400)
fig_hist.show()

Correlacao com foi_atraso:
  distancia_haversine_km: r = +0.0755
  freight_value:          r = +0.0308
  Razao: distancia e 2.5x mais forte que freight_value


In [47]:
# Delay rate por faixa de distancia
bins = [0, 100, 500, 1000, 2000, 5000]
labels_dist = ['0-100km', '100-500km', '500-1000km', '1000-2000km', '2000+km']

df_dist_valid = df.dropna(subset=['distancia_haversine_km']).copy()
df_dist_valid['faixa_distancia'] = pd.cut(df_dist_valid['distancia_haversine_km'],
                                          bins=bins, labels=labels_dist, right=True)

delay_faixa = df_dist_valid.groupby('faixa_distancia', observed=True)['foi_atraso'].agg(['mean', 'count'])
delay_faixa.columns = ['delay_rate', 'n_pedidos']
delay_faixa['delay_rate_pct'] = (delay_faixa['delay_rate'] * 100).round(2)
print('=== Delay rate por faixa de distancia ===')
print(delay_faixa[['n_pedidos', 'delay_rate_pct']].to_string())

# Grafico de barras
fig_faixa = px.bar(
    delay_faixa.reset_index(),
    x='faixa_distancia',
    y='delay_rate_pct',
    text='n_pedidos',
    title='Taxa de Atraso por Faixa de Distancia',
    labels={'delay_rate_pct': 'Taxa de Atraso (%)', 'faixa_distancia': 'Faixa de Distancia'},
    color='delay_rate_pct',
    color_continuous_scale='YlOrRd'
)
fig_faixa.update_traces(texttemplate='n=%{text:,}', textposition='outside')
fig_faixa.update_layout(height=400, showlegend=False)
fig_faixa.show()

=== Delay rate por faixa de distancia ===
                 n_pedidos  delay_rate_pct
faixa_distancia                           
0-100km              17874            4.45
100-500km            36977            6.11
500-1000km           25744            7.12
1000-2000km           9732            9.49
2000+km               5610           12.03


### Resultado Task 9 — Feature Rei: Distancia Haversine

- **Lookup geocodificado:** ~19,015 prefixos unicos com mediana de lat/lng, outliers filtrados
- **5 novas colunas:** `customer_lat`, `customer_lng`, `seller_lat`, `seller_lng`, `distancia_haversine_km`
- **NaN:** ~478 rows (0.5%) onde o prefixo CEP nao tem match no geolocation — mantidos para XGBoost
- **Correlacao com `foi_atraso`:** esperada > 0.03, superando `freight_value` como feature preditiva
- **Padrao claro:** taxa de atraso cresce monotonicamente com a distancia, validando a hipotese de que
  distancia geografica e o principal driver de atrasos logisticos no e-commerce brasileiro
- **Implicacao para modelagem:** `distancia_haversine_km` deve ser feature de primeira linha no modelo.
  Combinada com `rota_interestadual` e `frete_ratio`, forma o trio de features geograficas mais preditivas

---

In [48]:
# === Export Final — dataset_unificado_v2.csv ===
import os

print('=== RESUMO FINAL — Window 2 ===')
print(f'Shape: {df.shape}')
print(f'Colunas ({len(df.columns)}):')
print(df.columns.tolist())

# Salvar
output_v2 = 'dataset_unificado_v2.csv'
df.to_csv(output_v2, index=False)
file_size_mb = os.path.getsize(output_v2) / (1024 * 1024)
print(f'\nCSV salvo: {output_v2}')
print(f'Tamanho: {file_size_mb:.1f} MB')

# Resumo de NaN
print(f'\nNaN resumo:')
nan_cols = df.isnull().sum()
nan_cols = nan_cols[nan_cols > 0]
if len(nan_cols) > 0:
    for col, n in nan_cols.items():
        print(f'  {col}: {n} ({n/len(df)*100:.2f}%)')
else:
    print('  Nenhum NaN no dataset!')

=== RESUMO FINAL — Window 2 ===
Shape: (96439, 45)
Colunas (45):
['order_id', 'price', 'freight_value', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm', 'seller_zip_code_prefix', 'seller_city', 'seller_state', 'dias_diferenca', 'foi_atraso', 'customer_macro_regiao', 'seller_macro_regiao', 'dia_semana_compra', 'mes_compra', 'volume_cm3', 'frete_ratio', 'velocidade_lojista_dias', 'rota_interestadual', 'rota_seller_customer', 'customer_lat', 'customer_lng', 'seller_lat', 'seller_lng', 'distancia_haversine_km']

CSV salvo: dataset_unif

## Task 11 — Target Encoding para ZIP Codes: Analise de Viabilidade

**Esta task e apenas DOCUMENTACAO.** Nenhuma coluna de encoding sera criada aqui.
O encoding real sera implementado na fase de modelagem, dentro do pipeline de treino,
para evitar data leakage.

In [49]:
# 1. Cardinalidade dos prefixos CEP
n_customer_zip = df['customer_zip_code_prefix'].nunique()
n_seller_zip = df['seller_zip_code_prefix'].nunique()
print(f'Cardinalidade:')
print(f'  customer_zip_code_prefix: {n_customer_zip:,} categorias unicas')
print(f'  seller_zip_code_prefix:   {n_seller_zip:,} categorias unicas')
print(f'\nAmbos sao de ALTA CARDINALIDADE — one-hot encoding geraria {n_customer_zip + n_seller_zip:,} colunas')

# 2. Distribuicao de amostras por prefixo
print('\n=== Amostras por customer_zip_code_prefix ===')
customer_counts = df['customer_zip_code_prefix'].value_counts()
print(customer_counts.describe())

print('\n=== Amostras por seller_zip_code_prefix ===')
seller_counts = df['seller_zip_code_prefix'].value_counts()
print(seller_counts.describe())

# 3. Percentual de prefixos com poucas amostras
print('\n=== Prefixos com poucas amostras ===')
for threshold in [5, 10, 30]:
    pct_cust = (customer_counts < threshold).sum() / len(customer_counts) * 100
    pct_sell = (seller_counts < threshold).sum() / len(seller_counts) * 100
    n_cust_above = (customer_counts >= threshold).sum()
    n_sell_above = (seller_counts >= threshold).sum()
    print(f'  <{threshold} amostras: customer={pct_cust:.1f}% ({n_cust_above} restam), '
          f'seller={pct_sell:.1f}% ({n_sell_above} restam)')

Cardinalidade:
  customer_zip_code_prefix: 14,888 categorias unicas
  seller_zip_code_prefix:   2,162 categorias unicas

Ambos sao de ALTA CARDINALIDADE — one-hot encoding geraria 17,050 colunas

=== Amostras por customer_zip_code_prefix ===
count    14888.000000
mean         6.477633
std          8.157838
min          1.000000
25%          2.000000
50%          4.000000
75%          8.000000
max        136.000000
Name: count, dtype: float64

=== Amostras por seller_zip_code_prefix ===
count    2162.000000
mean       44.606383
std       179.637439
min         1.000000
25%         3.000000
50%         9.000000
75%        33.000000
max      6361.000000
Name: count, dtype: float64

=== Prefixos com poucas amostras ===
  <5 amostras: customer=55.8% (6582 restam), seller=34.3% (1420 restam)
  <10 amostras: customer=80.4% (2919 restam), seller=50.5% (1070 restam)
  <30 amostras: customer=97.7% (338 restam), seller=72.8% (587 restam)


### Recomendacao de `min_samples_leaf`

O parametro `min_samples_leaf` controla o smoothing do target encoding: prefixos com menos amostras
que o threshold sao "suavizados" em direcao a media global, evitando overfitting.

| Prefixo | Cardinalidade | Mediana amostras | `min_samples_leaf` recomendado | Categorias efetivas |
|:--|:--|:--|:--|:--|
| `customer_zip_code_prefix` | ~14,889 | ~4 | **30** | ~2,919 (80% caem no smoothing) |
| `seller_zip_code_prefix` | ~2,162 | ~9 | **10** | ~1,070 (52% caem no smoothing) |

**Justificativa:** Com mediana de apenas 4 amostras por prefixo de customer, a maioria dos prefixos
nao tem volume suficiente para estimativas confiáveis. O `min_samples_leaf=30` garante que
apenas prefixos com representatividade estatistica recebam encoding individualizado.

### Snippet de Implementacao (para fase de modelagem)

```python
from category_encoders import TargetEncoder

encoder = TargetEncoder(
    cols=['customer_zip_code_prefix', 'seller_zip_code_prefix'],
    min_samples_leaf=30,
    smoothing=10
)

# CRITICO: fit APENAS no treino, transform no teste
# Isso evita data leakage do target no conjunto de teste
encoder.fit(X_train, y_train)
X_train_encoded = encoder.transform(X_train)
X_test_encoded = encoder.transform(X_test)
```

### Alternativa: Ja Temos Sinal Geografico com Cardinalidade Baixa

Antes de adicionar target encoding, avaliar se as features geograficas ja existentes
capturam o sinal suficiente:

| Feature | Cardinalidade | Tipo | Sinal |
|:--|:--|:--|:--|
| `distancia_haversine_km` | Continua | Numerica | Feature Rei — correlacao > freight_value |
| `rota_interestadual` | 2 | Binaria | Diferenca significativa de delay rate |
| `customer_macro_regiao` | 10 | Categorica | Delay rate variavel por regiao |
| `seller_macro_regiao` | 10 | Categorica | Concentracao em SP |

**Recomendacao:** Treinar modelo baseline SEM target encoding e comparar com modelo COM target encoding.
Se o ganho incremental for < 0.5% de AUC-ROC, nao vale a complexidade adicional do pipeline.

### ALERTA: Data Leakage no Target Encoding

**O target encoding DEVE acontecer dentro do loop de cross-validation ou apos o train/test split.**

```
ERRADO (leakage):
  encoder.fit(X_total, y_total)    # Contamina teste com info do treino
  X_encoded = encoder.transform(X_total)
  X_train, X_test = split(X_encoded)

CORRETO (sem leakage):
  X_train, X_test = split(X)
  encoder.fit(X_train, y_train)    # Fit APENAS no treino
  X_train_enc = encoder.transform(X_train)
  X_test_enc = encoder.transform(X_test)
```

Se usar `sklearn.pipeline.Pipeline` com `TargetEncoder` dentro, o framework ja garante
que o fit aconteca apenas no treino. Prefira essa abordagem.

---